# Explore the Vietnamese Medical Q&A Dataset

**Objective:** This notebook performs initial data exploration (EDA) steps on the `hungnm/vietnamese-medical-qa` dataset.

**Key steps:**
1. Install and import the necessary libraries.
2. Load and view basic information of the dataset.
3. Statistical analysis and visualization of question and answer length.
4. Analyze the most common words to understand the main topics.
5. Check data quality (missing values, duplicates).
6. Summarize conclusions and recommendations for next steps.

In [ ]:
# Cell 1: Install the necessary libraries
!pip install -q datasets pandas matplotlib seaborn pyvi wordcloud

In [ ]:
# Cell 2: Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from collections import Counter
import re
from pyvi import ViTokenizer
from wordcloud import WordCloud

# Set style for plots
sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'sans-serif' # Make sure the font displays in Vietnamese

## 1. Download and View Basic Information of Dataset

In [ ]:
# Cell 3: Load dataset from Hugging Face
dataset_name = "hungnm/vietnamese-medical-qa"
dataset = load_dataset(dataset_name, split="train")

print("Dataset information:")
print(dataset)

print("\nColumns in the dataset:")
print(dataset.features)

In [ ]:
# Cell 4: View some random data samples
print("--- 5 Random Data Samples ---")
sample_data = dataset.shuffle(seed=42).select(range(5))

for i, sample in enumerate(sample_data):
    print(f"\n--- Sample {i+1} ---")
    print(f"❓ Question: {sample['question']}")
    print(f"💬 Answer: {sample['answer']}")

## 2. Text Length Analysis

In [ ]:
# Cell 5: Convert dataset to Pandas DataFrame for easy analysis
df = dataset.to_pandas()

# Calculate the length (number of words) of the question and answer
df['question_length'] = df['question'].apply(lambda x: len(str(x).split()))
df['answer_length'] = df['answer'].apply(lambda x: len(str(x).split()))

print("Statistics about the length of questions and answers:")
display(df[['question_length', 'answer_length']].describe())

In [ ]:
# Cell 6: Visualizing the length distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Distribution of Question and Answer Lengths', fontsize=20)

# Histogram for question length
sns.histplot(df['question_length'], bins=50, ax=axes[0, 0], kde=True)
axes[0, 0].set_title('Distribution of Question Length (Number of Words)')
axes[0, 0].set_xlabel('Number of Words')
axes[0, 0].set_ylabel('Frequency')

# Box plot for question length
sns.boxplot(x=df['question_length'], ax=axes[0, 1])
axes[0, 1].set_title('Answer Length Boxplot')
axes[0, 1].set_xlabel('Number of Words')

# Histogram for answer length
sns.histplot(df['answer_length'], bins=50, ax=axes[1, 0], kde=True, color='green')
axes[1, 0].set_title('Answer Length Distribution (Number of Words)')
axes[1, 0].set_xlabel('Number of Words')
axes[1, 0].set_ylabel('Frequency')

# Box plot for answer length
sns.boxplot(x=df['answer_length'], ax=axes[1, 1], color='green')
axes[1, 1].set_title('Answer Length Boxplot')
axes[1, 1].set_xlabel('Number of words')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## 3. Text Content Analysis

We will look at the most common words and phrases in the responses to better understand the main medical topics in the dataset.

In [ ]:
# Cell 7: Tìm các từ phổ biến nhất

# Danh sách stop words tiếng Việt cơ bản
vietnamese_stopwords = set([
    'bị', 'bởi', 'cả', 'các', 'cái', 'cần', 'càng', 'chỉ', 'chiếc', 'cho', 'chứ', 'chưa', 'có', 'có_thể', 'cứ',
    'của', 'cùng', 'cũng', 'đã', 'đang', 'đây', 'để', 'đến', 'đó', 'được', 'gì', 'khi', 'không', 'là', 'lại',
    'lên', 'lúc', 'mà', 'mỗi', 'một', 'nên', 'nếu', 'ngay', 'nhiều', 'như', 'nhưng', 'những', 'nơi', 'nữa',
    'phải', 'qua', 'ra', 'rằng', 'rất', 'rồi', 'sau', 'sẽ', 'so', 'sự', 'tại', 'theo', 'thì', 'trên', 'trước',
    'từ', 'từng', 'và', 'vẫn', 'vào', 'vậy', 'vì', 'việc', 'với', 'vừa', 'ạ', 'bạn', 'em', 'anh', 'chị', 'cháu'
])

def get_top_n_words(corpus, n=20):
    # Combine all answers into one big text string
    text = ' '.join(corpus)
    
    # Tokenize and convert to lowercase
    text = ViTokenizer.tokenize(text.lower())
    
    # Find all words, remove punctuation and non-words
    words = re.findall(r'\b[a-zA-Z_]+\b', text)
    
    # Filter out stop words
    filtered_words = [word for word in words if word not in vietnamese_stopwords]
    
    # Count frequency and get n most common words
    word_counts = Counter(filtered_words)
    return word_counts.most_common(n)

top_words = get_top_n_words(df['answer'], n=25)

# Visualization
top_words_df = pd.DataFrame(top_words, columns=['word', 'count'])

plt.figure(figsize=(12, 8))
sns.barplot(x='count', y='word', data=top_words_df, palette='viridis')
plt.title('25 Most Common Words in Answers (stop words removed)', fontsize=16)
plt.xlabel('Tần suất')
plt.ylabel('Từ')
plt.show()

In [ ]:
# Cell 8: Create a Word Cloud to visualize common words

text = ' '.join(df['answer'])
tokenized_text = ViTokenizer.tokenize(text.lower())

wordcloud = WordCloud(
    width=800,
    height=400,
    background_color='white',
    stopwords=vietnamese_stopwords,
    min_font_size=10
).generate(tokenized_text)

plt.figure(figsize=(15, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.title("Word Cloud of Answers", fontsize=18)
plt.show()

## 4. Data Quality Check

In [ ]:
# Cell 9: Check for missing values ​​(null) and empty strings
print("Check for missing values ​​(Null/NaN):")
print(df.isnull().sum())

empty_questions = df[df['question'].str.strip() == ''].shape[0]
empty_answers = df[df['answer'].str.strip() == ''].shape[0]

print(f"\nNumber of empty questions (containing only spaces): {empty_questions}")
print(f"Number of empty answers (containing only spaces): {empty_answers}")

# Cell 10: Check for duplicate rows
duplicate_rows = df.duplicated().sum()
print(f"\nNumber of completely duplicated data rows: {duplicate_rows}")

## 5. Summary and Conclusion

Based on the above analysis, we can draw some conclusions:

1. **Overview:** The dataset contains 9335 medical question-answer pairs in Vietnamese, including two main columns `question` and `answer`.

2. **Length:**
- The questions are usually quite short, mostly under 50 words.

- The answers have more diverse lengths, mostly concentrated in the range of 50-250 words. There are some exceptional answers that are very long (over 500 words).

- **Implication:** Choosing `max_seq_length` as 512 or 1024 for fine-tuning is reasonable to cover most of the samples.

3. **Content:**
- The most common words such as `doctor`, `treatment`, `condition`, `symptom`, `examination`, `baby`, `health` show that the dataset focuses heavily on consultation, diagnosis and treatment of common medical problems.

- Many terms related to specific specialties such as `dermatology`, `sinus`, `stomach`, `reproductive` also appear.

4. **Data quality:**
- The dataset has quite good quality: **no missing values ​​(null)** and **no empty strings**.

- The number of duplicate rows is 0, showing that the data is quite clean.

**Next Step:** The data is ready to be preprocessed and fed into training pipelines (SFT, TVAFT, ReFT) without any complicated cleaning steps.